# 저작권 메타데이터 자동 추출 시스템
**숭실대학교 데이터베이스 연구실**

PDF/이미지 문서에서 메타데이터를 자동으로 추출합니다.

**파이프라인:** OCR → LLM 추출 → NER 개체명 추출 → 통합 검증

---

### 사용 방법
1. **[1단계]** 환경 설정 셀을 실행합니다 (최초 1회, 약 5분)
2. **[2단계]** PDF 파일을 업로드합니다
3. **[3단계]** 추출을 실행합니다
4. **[4단계]** 결과를 확인하고 다운로드합니다

## [1단계] 환경 설정 (최초 1회 실행)

In [ ]:
#@title 1-1. 패키지 업로드 및 설치 {display-mode: "form"}
#@markdown **copyright_extraction_cli.tar.gz** 파일을 업로드하세요.
#@markdown
#@markdown 또는 Google Drive에서 로드할 수 있습니다.

import os

# Option A: Upload from local
use_drive = False  #@param {type:"boolean"}
drive_path = "/content/drive/MyDrive/copyright_extraction_cli.tar.gz"  #@param {type:"string"}

if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp "{drive_path}" /content/
    print("Google Drive에서 복사 완료")
else:
    from google.colab import files
    print("copyright_extraction_cli.tar.gz 파일을 업로드하세요...")
    uploaded = files.upload()
    print(f"업로드 완료: {list(uploaded.keys())}")

# Extract
!cd /content && tar xzf copyright_extraction_cli.tar.gz
print("\n압축 해제 완료!")
!ls /content/copyright_extraction_cli/

In [ ]:
#@title 1-2. 의존성 설치 (약 3-5분) {display-mode: "form"}

!cd /content/copyright_extraction_cli && pip install -q -r requirements.txt

# Configure Python path
import site, sys
pth_path = os.path.join(site.getsitepackages()[0], 'copyright-metadata.pth')
with open(pth_path, 'w') as f:
    f.write('/content/copyright_extraction_cli/api\n')

# Add to current session
sys.path.insert(0, '/content/copyright_extraction_cli/api')

# Verify
!cd /content/copyright_extraction_cli && python extract.py --list-models
print("\n환경 설정 완료!")

## [2단계] PDF 파일 업로드

In [ ]:
#@title 2. PDF 파일 업로드 {display-mode: "form"}
#@markdown 추출할 PDF 파일을 업로드하세요. (여러 파일 가능)

from google.colab import files
import shutil

# Create input directory
input_dir = '/content/input_documents'
os.makedirs(input_dir, exist_ok=True)

print("PDF 파일을 업로드하세요...")
uploaded = files.upload()

for name, data in uploaded.items():
    path = os.path.join(input_dir, name)
    with open(path, 'wb') as f:
        f.write(data)
    size_mb = len(data) / (1024 * 1024)
    print(f"  저장: {name} ({size_mb:.2f} MB)")

print(f"\n총 {len(uploaded)}개 파일 업로드 완료")
print(f"저장 위치: {input_dir}")

## [3단계] 메타데이터 추출 실행

In [ ]:
#@title 3. 추출 설정 및 실행 {display-mode: "form"}

#@markdown ### 추출 설정
document_type = "\uAE30\uD0C0\uBB38\uC11C" #@param ["계약서", "동의서", "저작재산권 양도동의서", "공공저작물 자유이용허락 동의서", "기타문서"]
pipeline_stages = "all" #@param ["all", "ocr", "ocr+ner", "ocr+llm", "ocr+llm+ner"]
ocr_provider = "alibaba" #@param ["alibaba", "google", "mistral"]

output_dir = '/content/extraction_results'

# Count files
input_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.pdf', '.jpg', '.jpeg', '.png', '.tiff', '.tif', '.gif', '.bmp'))]
print(f"추출 대상: {len(input_files)}개 파일")
for f in input_files:
    print(f"  - {f}")

print(f"\n문서 유형: {document_type}")
print(f"파이프라인: {pipeline_stages}")
print(f"OCR 제공자: {ocr_provider}")
print(f"\n추출 시작...\n")

# Determine input path
if len(input_files) == 1:
    file_path = os.path.join(input_dir, input_files[0])
else:
    file_path = input_dir

# Run extraction with real-time output
import subprocess
process = subprocess.Popen(
    ['python', 'extract.py', file_path, '-t', document_type,
     '-s', pipeline_stages, '--ocr-provider', ocr_provider, '-o', output_dir],
    cwd='/content/copyright_extraction_cli',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print("\n추출 완료!")
else:
    print(f"\n오류 발생 (종료 코드: {process.returncode})")

## [4단계] 결과 확인 및 다운로드

In [ ]:
#@title 4-1. 결과 확인 {display-mode: "form"}

import json
from pathlib import Path

results_path = Path(output_dir)
if not results_path.exists():
    print("결과가 없습니다. 3단계를 먼저 실행하세요.")
else:
    for doc_dir in sorted(results_path.iterdir()):
        if doc_dir.is_dir():
            print(f"\n{'='*70}")
            print(f"  문서: {doc_dir.name}")
            print(f"{'='*70}")

            # OCR text preview
            ocr_file = doc_dir / 'ocr_text.txt'
            if ocr_file.exists():
                text = ocr_file.read_text(encoding='utf-8')
                print(f"\n  [OCR] {len(text)} 문자 추출")
                print(f"  처음 200자: {text[:200]}...")

            # Consolidated metadata
            con_file = doc_dir / 'consolidated_metadata.json'
            if con_file.exists():
                meta = json.loads(con_file.read_text(encoding='utf-8'))
                print(f"\n  [통합 메타데이터] {len(meta)} 필드")
                for k, v in meta.items():
                    val = str(v)[:60] if v else 'null'
                    print(f"    {k}: {val}")

            # NER entities
            ner_file = doc_dir / 'ner_entities.json'
            if ner_file.exists():
                ner = json.loads(ner_file.read_text(encoding='utf-8'))
                total = ner.get('total_entities', 0)
                print(f"\n  [NER] {total} 엔티티 추출")

            # File list
            print(f"\n  생성된 파일:")
            for f in sorted(doc_dir.glob('*')):
                if f.is_file() and not f.name.startswith('_'):
                    print(f"    {f.name} ({f.stat().st_size:,} bytes)")

In [ ]:
#@title 4-2. 결과 다운로드 (ZIP) {display-mode: "form"}

import shutil
from google.colab import files

# Create zip of results
zip_path = '/content/extraction_results'
if os.path.exists(zip_path):
    shutil.make_archive('/content/extraction_results_download', 'zip', zip_path)
    print("결과 ZIP 파일 생성 완료")
    files.download('/content/extraction_results_download.zip')
else:
    print("결과가 없습니다. 3단계를 먼저 실행하세요.")

---

## 개별 파일 보기 (선택사항)

In [ ]:
#@title OCR 텍스트 전문 보기 {display-mode: "form"}

from pathlib import Path

results_path = Path('/content/extraction_results')
for doc_dir in sorted(results_path.iterdir()):
    if doc_dir.is_dir():
        ocr_file = doc_dir / 'ocr_text.txt'
        if ocr_file.exists():
            print(f"=== {doc_dir.name} ===")
            print(ocr_file.read_text(encoding='utf-8'))
            print()

In [ ]:
#@title LLM 메타데이터 보기 {display-mode: "form"}

import json
from pathlib import Path

results_path = Path('/content/extraction_results')
for doc_dir in sorted(results_path.iterdir()):
    if doc_dir.is_dir():
        llm_file = doc_dir / 'llm_metadata.json'
        if llm_file.exists():
            print(f"=== {doc_dir.name} ===")
            data = json.loads(llm_file.read_text(encoding='utf-8'))
            print(json.dumps(data.get('metadata', {}), ensure_ascii=False, indent=2))
            print()

In [ ]:
#@title NER 엔티티 상세 보기 {display-mode: "form"}

import json, glob
from pathlib import Path

results_path = Path('/content/extraction_results')
for doc_dir in sorted(results_path.iterdir()):
    if doc_dir.is_dir():
        # Find internal NER result with actual entity values
        internal_files = list(doc_dir.rglob('*_entities.json'))
        for f in internal_files:
            if '_results' in str(f):
                print(f"=== {doc_dir.name} ===")
                data = json.loads(f.read_text(encoding='utf-8'))
                entities = data.get('entities', {})
                for etype, vals in sorted(entities.items()):
                    if isinstance(vals, list):
                        print(f"  {etype} ({len(vals)}): {vals}")
                print()
                break

---

### 모델 정보

| 단계 | 모델 | 실행 위치 |
|------|------|-----------|
| OCR | Qwen3-VL-235B | Alibaba Cloud API |
| LLM 추출 | Qwen3.5-122B | Alibaba Cloud API |
| NER 개체명 | KLUE-RoBERTa-Large | 로컬 (Colab CPU/GPU) |
| 통합 검증 | Qwen3.5-122B | Alibaba Cloud API |

### 문의
숭실대학교 데이터베이스 연구실